# 04 — Feature Engineering & Analytical Dataset

## 1. Introduction and objectives

Phase 1C (`03_data_cleaning_preparation.ipynb`) produced typed and flagged copies of the Olist tables in `data/processed/`. Phase 1D turns them into two **analytical datasets** for the next CallVerse phases:

1. **Order-level table**: one row per order, combining the order with its customer, items, payments and latest review as engineered features.
2. **Daily activity table**: one row per calendar day with counts of e-commerce events (orders placed, deliveries, late deliveries, reviews, …). The forecasting phase will need it.

Every feature is tagged with the **stage at which it becomes known**: at purchase, at carrier handoff, at delivery, at review, or only with hindsight at the end of the data. A later simulated contact at time *t* may only use features known before *t*; using later information would be **leakage**.

> **These are e-commerce events, not contacts.** Olist has no contact logs, contact reasons, call durations, queue times or agent assignments. A late delivery or a low review score may later be used to *simulate* contacts under documented assumptions, but no contact variable is created here. The daily table counts orders and deliveries, never contacts.

## 2. Imports and project paths

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR, "exists:", PROCESSED_DIR.exists())
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(f"Processed-data directory missing: {PROCESSED_DIR}.")

Project root: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse
Processed data directory: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse\data\processed exists: True


## 3. Load the prepared datasets

We read the Phase 1C Parquet files, which keep datetime and boolean types. If they are missing, run `notebooks/03_data_cleaning_preparation.ipynb` first. The product-category translation table is read (read-only) from `data/raw/olist/` because Phase 1C did not prepare it.

In [2]:
PREPARED_FILES = {
    "orders": "olist_orders_prepared.parquet",
    "order_items": "olist_order_items_prepared.parquet",
    "customers": "olist_customers_prepared.parquet",
    "payments": "olist_payments_prepared.parquet",
    "reviews": "olist_reviews_prepared.parquet",
    "products": "olist_products_prepared.parquet",
    "sellers": "olist_sellers_prepared.parquet",
}
TRANSLATION_FILE = DATA_DIR / "product_category_name_translation.csv"

missing_files = []
for file_name in PREPARED_FILES.values():
    if not (PROCESSED_DIR / file_name).exists():
        missing_files.append(file_name)
if not TRANSLATION_FILE.exists():
    missing_files.append(str(TRANSLATION_FILE))
if missing_files:
    raise FileNotFoundError(
        f"Missing input files: {missing_files}. "
        "Run notebooks/03_data_cleaning_preparation.ipynb first and check data/raw/olist/."
    )

prepared = {}
for dataset_name, file_name in PREPARED_FILES.items():
    prepared[dataset_name] = pd.read_parquet(PROCESSED_DIR / file_name)
category_translation = pd.read_csv(TRANSLATION_FILE)

orders = prepared["orders"]
order_items = prepared["order_items"]
customers = prepared["customers"]
payments = prepared["payments"]
reviews = prepared["reviews"]
products = prepared["products"]
sellers = prepared["sellers"]

load_summary = []
for dataset_name, dataset in prepared.items():
    load_summary.append({"dataset": dataset_name, "rows": len(dataset), "columns": len(dataset.columns)})
load_summary.append({"dataset": "category_translation", "rows": len(category_translation),
                     "columns": len(category_translation.columns)})
display(pd.DataFrame(load_summary))

,dataset,rows,columns
0,orders,99441,21
1,order_items,112650,7
2,customers,99441,5
3,payments,103886,8
4,reviews,99224,12
5,products,32951,11
6,sellers,3095,4
7,category_translation,71,2


## 4. Design decisions

| Topic | Decision |
|---|---|
| Grain | One row per order (`order_id`). Items, payments and reviews are **aggregated** to the order, so no order is repeated. The prepared tables keep the detailed rows. |
| Daily table | One row per calendar day from the first purchase to the last recorded order event, including days with no events (count 0). |
| Analysis window | **No order is removed.** `in_analysis_window` marks purchases from 1 January 2017 to 31 August 2018; the months before and after are too sparse (section 10). |
| Contradictory timestamps | A duration is left **empty** when the two timestamps it uses were flagged as contradictory in Phase 1C (for example, carrier handoff before approval). The order's other features are kept. |
| Several reviews per order | The **latest** review (latest answer, then latest creation) represents the order; `review_count` records how many rows exist. |
| Primary item | Where one value per order is needed (category, seller), we use the **most expensive item line**; ties go to the lowest `order_item_id`. |
| Leakage | Each feature has an `available_at` stage in the feature catalog (section 11). |
| Missing values | Not imputed. A feature is empty when its source is missing or the event did not happen. |

`available_at` stages used in this notebook:

| Stage | Meaning |
|---|---|
| `identifier` | Keys and labels; not a predictive feature. |
| `purchase` | Known when the order is placed (order contents, payment, promised delivery date, customer history up to this order). |
| `approval` | Known once the payment is approved. |
| `carrier_handoff` | Known once the seller hands the parcel to the carrier. |
| `delivery` | Known once the customer receives the order. |
| `review` | Known once the customer writes a review. |
| `end_of_data` | Only known with hindsight at the end of the dataset (final status, overdue orders, censoring). |

## 5. Order base: orders, customers and purchase calendar

### 5.1 Join each order with its customer

`customer_id` is unique in both tables (one customer row per order), so this join cannot multiply rows. `validate="one_to_one"` makes pandas raise an error if that ever stops being true.

In [3]:
order_features = orders.merge(customers, on="customer_id", how="left", validate="one_to_one")
print("Orders:", len(orders), "| rows after joining customers:", len(order_features))

Orders: 99441 | rows after joining customers: 99441


### 5.2 Purchase calendar features

These describe *when* the order was placed and are known at purchase. They matter later because contact volume in a real contact center follows daily and weekly patterns.

In [4]:
purchase = order_features["order_purchase_timestamp"]
order_features["purchase_date"] = purchase.dt.normalize()
order_features["purchase_year_month"] = purchase.dt.strftime("%Y-%m")
order_features["purchase_day_of_week"] = purchase.dt.day_name()
order_features["purchase_hour"] = purchase.dt.hour
order_features["purchase_is_weekend"] = purchase.dt.dayofweek >= 5  # Saturday = 5, Sunday = 6

display(order_features["purchase_day_of_week"].value_counts().to_frame("orders"))

,orders
purchase_day_of_week,
Monday,16196
Tuesday,15963
Wednesday,15552
Thursday,14761
Friday,14122
Sunday,11960
Saturday,10887


### 5.3 Customer history

`customer_order_number` is the position of this order among all orders of the same person (`customer_unique_id`), sorted by purchase time: 1 for a first order, 2 for a second, and so on. It uses only *earlier* orders, so it is known at purchase. Orders of the same person at the exact same time are ordered by `order_id`.

In [5]:
sorted_orders = order_features.sort_values(["customer_unique_id", "order_purchase_timestamp", "order_id"])
# cumcount numbers rows within each person from 0; the result is aligned back by index
order_features["customer_order_number"] = sorted_orders.groupby("customer_unique_id").cumcount() + 1
order_features["customer_is_repeat_buyer"] = order_features["customer_order_number"] > 1

display(order_features["customer_order_number"].value_counts().sort_index().to_frame("orders"))

,orders
customer_order_number,
1,96096
2,2997
3,252
4,49
5,19
6,11
7,5
8,2
9,2


## 6. Item features

First we add product, category and seller information to each item line (each lookup is many-to-one, so item rows are not multiplied). Then we aggregate the items to one row per order.

- **Totals:** item count, distinct products / sellers / categories, price, freight, order value (price + freight), freight share.
- **Weight:** total product weight; empty if any item of the order has no weight.
- **Primary item:** category (Portuguese and English) and seller of the most expensive item line.
- **Shipping deadline:** the latest `shipping_limit_date` among the order's items, i.e. the date by which the seller had to hand the order to the carrier.

Two categories have no English translation (Phase 1B); their English name stays empty rather than being invented.

In [6]:
item_details = order_items.merge(products[["product_id", "product_category_name", "product_weight_g"]],
                                 on="product_id", how="left", validate="many_to_one")
item_details = item_details.merge(category_translation, on="product_category_name", how="left", validate="many_to_one")
item_details = item_details.merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one")
item_details["weight_is_missing"] = item_details["product_weight_g"].isna()
print("Item rows before and after lookups:", len(order_items), len(item_details))

item_summary = item_details.groupby("order_id").agg(
    item_count=("order_item_id", "size"),
    distinct_products=("product_id", "nunique"),
    distinct_sellers=("seller_id", "nunique"),
    distinct_categories=("product_category_name", "nunique"),
    items_price_total=("price", "sum"),
    items_freight_total=("freight_value", "sum"),
    items_weight_g_total=("product_weight_g", "sum"),
    items_with_missing_weight=("weight_is_missing", "sum"),
    latest_shipping_limit_date=("shipping_limit_date", "max"),
).reset_index()

item_summary["order_value"] = item_summary["items_price_total"] + item_summary["items_freight_total"]
item_summary["freight_share"] = item_summary["items_freight_total"] / item_summary["order_value"]
# A total weight is only meaningful if every item has a weight
item_summary["items_weight_g_total"] = item_summary["items_weight_g_total"].mask(item_summary["items_with_missing_weight"] > 0)
item_summary = item_summary.drop(columns=["items_with_missing_weight"])

# Primary item: most expensive line of each order, ties broken by the lowest order_item_id
primary_items = item_details.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True])
primary_items = primary_items.drop_duplicates(subset=["order_id"], keep="first")
primary_items = primary_items[["order_id", "product_category_name", "product_category_name_english", "seller_id", "seller_state"]]
primary_items = primary_items.rename(columns={
    "product_category_name": "primary_category",
    "product_category_name_english": "primary_category_english",
    "seller_id": "primary_seller_id",
    "seller_state": "primary_seller_state",
})

order_features = order_features.merge(item_summary, on="order_id", how="left", validate="one_to_one")
order_features = order_features.merge(primary_items, on="order_id", how="left", validate="one_to_one")
print("Rows after adding item features:", len(order_features))

display(order_features[["item_count", "distinct_products", "distinct_sellers", "items_price_total",
                        "items_freight_total", "order_value", "freight_share", "items_weight_g_total"]].describe().round(2))

Item rows before and after lookups: 112650 112650
Rows after adding item features: 99441


,item_count,distinct_products,distinct_sellers,items_price_total,items_freight_total,order_value,freight_share,items_weight_g_total
count,98666.00,98666.00,98666.00,98666.00,98666.00,98666.00,98666.00,98650.00
mean,1.14,1.04,1.01,137.75,22.82,160.58,0.21,2390.42
std,0.54,0.23,0.12,210.65,21.65,220.47,0.13,4773.53
min,1.00,1.00,1.00,0.85,0.00,9.59,0.00,0.00
25%,1.00,1.00,1.00,45.90,13.85,61.98,0.12,300.00
50%,1.00,1.00,1.00,86.90,17.17,105.29,0.18,750.00
75%,1.00,1.00,1.00,149.90,24.04,176.87,0.28,2068.75
max,21.00,8.00,5.00,13440.00,1794.96,13664.08,0.96,184400.00


In [7]:
# Does the customer live in the same state as the primary seller? Empty when the order has no items.
same_state = (order_features["customer_state"] == order_features["primary_seller_state"]).astype("boolean")
same_state[order_features["primary_seller_state"].isna()] = pd.NA
order_features["customer_seller_same_state"] = same_state

print("Primary category with no English name (category present):",
      int((order_features["primary_category"].notna() & order_features["primary_category_english"].isna()).sum()))
display(order_features["primary_category_english"].value_counts().head(10).to_frame("orders"))
display(order_features["customer_seller_same_state"].value_counts(dropna=False).to_frame("orders"))

Primary category with no English name (category present): 22


,orders
primary_category_english,
bed_bath_table,9328
health_beauty,8801
sports_leisure,7683
computers_accessories,6670
furniture_decor,6330
housewares,5821
watches_gifts,5607
telephony,4179
auto,3880


,orders
customer_seller_same_state,
False,63198
True,35468
<NA>,775


## 7. Payment features

Aggregated per order: number of payment rows, total paid, largest number of instalments, whether a voucher was used, and the **primary payment type** (the type of the largest payment; ties go to the first payment step). `payment_minus_order_value` compares the amount paid with price + freight; Phase 1B found 249 orders where they differ by more than 1 BRL.

In [8]:
payment_details = payments.copy()
payment_details["is_voucher"] = payment_details["payment_type"] == "voucher"

payment_summary = payment_details.groupby("order_id").agg(
    payment_count=("payment_sequential", "size"),
    payment_total=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
    uses_voucher=("is_voucher", "any"),
).reset_index()

primary_payments = payment_details.sort_values(["order_id", "payment_value", "payment_sequential"], ascending=[True, False, True])
primary_payments = primary_payments.drop_duplicates(subset=["order_id"], keep="first")
primary_payments = primary_payments[["order_id", "payment_type"]].rename(columns={"payment_type": "primary_payment_type"})

order_features = order_features.merge(payment_summary, on="order_id", how="left", validate="one_to_one")
order_features = order_features.merge(primary_payments, on="order_id", how="left", validate="one_to_one")
order_features["payment_minus_order_value"] = order_features["payment_total"] - order_features["order_value"]
print("Rows after adding payment features:", len(order_features))

display(order_features["primary_payment_type"].value_counts(dropna=False).to_frame("orders"))
display(order_features[["payment_count", "payment_total", "max_installments", "payment_minus_order_value"]].describe().round(2))

Rows after adding payment features: 99441


,orders
primary_payment_type,
credit_card,74975
boleto,19784
voucher,3151
debit_card,1527
not_defined,3
NaN,1


,payment_count,payment_total,max_installments,payment_minus_order_value
count,99440.00,99440.00,99440.00,98665.00
mean,1.04,160.99,2.93,0.03
std,0.38,221.95,2.72,1.13
min,1.00,0.00,0.00,-51.62
25%,1.00,62.01,1.00,0.00
50%,1.00,105.29,2.00,0.00
75%,1.00,176.97,4.00,0.00
max,29.00,13664.08,24.00,182.81


## 8. Delivery timeline features

Durations between milestones describe how the order progressed. Following the design decision, each duration is **empty when its two timestamps were flagged as contradictory** in Phase 1C, and also empty when either timestamp is missing.

| Feature | Formula | Empty when | Known at |
|---|---|---|---|
| `purchase_to_approval_hours` | approval − purchase | `flag_approved_before_purchase` | approval |
| `approval_to_carrier_days` | carrier handoff − approval | `flag_carrier_before_approval` | carrier handoff |
| `carrier_to_delivery_days` | delivery − carrier handoff | `flag_delivered_before_carrier` | delivery |
| `purchase_to_delivery_days` | delivery − purchase | `flag_delivered_before_purchase` | delivery |
| `promised_delivery_days` | estimated date − purchase date (calendar days) | never (both always present) | purchase |
| `delivery_delay_days` | delivery date − estimated date (calendar days; negative = early) | not delivered | delivery |
| `carrier_after_shipping_limit` | carrier handoff later than the seller's shipping deadline | no handoff or no items | carrier handoff |

`delivered_after_estimated_date` (Phase 1C) is kept as the late-delivery indicator; it agrees with `delivery_delay_days > 0`.

In [9]:
purchase = order_features["order_purchase_timestamp"]
approved = order_features["order_approved_at"]
carrier =order_features["order_delivered_carrier_date"]
delivered = order_features["order_delivered_customer_date"]
estimate = order_features["order_estimated_delivery_date"]
seconds_per_day = 24 * 3600

purchase_to_approval_hours = (approved - purchase).dt.total_seconds() / 3600
approval_to_carrier_days = (carrier - approved).dt.total_seconds() / seconds_per_day
carrier_to_delivery_days = (delivered - carrier).dt.total_seconds() / seconds_per_day
purchase_to_delivery_days = (delivered - purchase).dt.total_seconds() / seconds_per_day

# mask() empties the value wherever the flag is True
order_features["purchase_to_approval_hours"] = purchase_to_approval_hours.mask(order_features["flag_approved_before_purchase"])
order_features["approval_to_carrier_days"] = approval_to_carrier_days.mask(order_features["flag_carrier_before_approval"])
order_features["carrier_to_delivery_days"] = carrier_to_delivery_days.mask(order_features["flag_delivered_before_carrier"])
order_features["purchase_to_delivery_days"] = purchase_to_delivery_days.mask(order_features["flag_delivered_before_purchase"])

order_features["promised_delivery_days"] = (estimate - order_features["purchase_date"]).dt.days
order_features["delivery_delay_days"] = (delivered.dt.normalize() - estimate).dt.days

# Seller handed over later than the shipping deadline? Empty when either date is missing.
late_handoff = (carrier > order_features["latest_shipping_limit_date"]).astype("boolean")
late_handoff[carrier.isna() | order_features["latest_shipping_limit_date"].isna()] = pd.NA
order_features["carrier_after_shipping_limit"] = late_handoff

duration_columns = ["purchase_to_approval_hours", "approval_to_carrier_days", "carrier_to_delivery_days",
                    "purchase_to_delivery_days", "promised_delivery_days", "delivery_delay_days"]
display(order_features[duration_columns].describe().round(2))

emptied_by_flag = pd.DataFrame({
    "duration": ["approval_to_carrier_days", "carrier_to_delivery_days"],
    "emptied_because_flagged": [int(order_features["flag_carrier_before_approval"].sum()),
                                int(order_features["flag_delivered_before_carrier"].sum())],
})
display(emptied_by_flag)
display(order_features["carrier_after_shipping_limit"].value_counts(dropna=False).to_frame("orders"))

,purchase_to_approval_hours,approval_to_carrier_days,carrier_to_delivery_days,purchase_to_delivery_days,promised_delivery_days,delivery_delay_days
count,99281.00,96285.00,96452.00,96476.00,99441.00,96476.00
mean,10.42,2.86,9.33,12.56,24.40,-11.88
std,26.04,3.50,8.76,9.55,8.83,10.18
min,0.00,0.00,0.00,0.53,2.00,-147.00
25%,0.22,0.90,4.10,6.77,19.00,-17.00
50%,0.34,1.85,7.10,10.22,24.00,-12.00
75%,14.58,3.62,12.03,15.72,29.00,-7.00
max,4509.18,125.76,205.19,209.63,156.00,188.00


,duration,emptied_because_flagged
0,approval_to_carrier_days,1359
1,carrier_to_delivery_days,23


,orders
carrier_after_shipping_limit,
False,88808
True,8849
<NA>,1784


In [10]:
# Consistency check: delivered_after_estimated_date (Phase 1C) should equal delivery_delay_days > 0
delay_is_positive = (order_features["delivery_delay_days"] > 0).astype("boolean")
delay_is_positive[order_features["delivery_delay_days"].isna()] = pd.NA
agrees = (delay_is_positive == order_features["delivered_after_estimated_date"]).fillna(True)
print("Orders where the two late-delivery definitions disagree:", int((~agrees).sum()))

# Simple status indicators (the final status is only known at the end of the data)
order_features["is_delivered"] = order_features["order_status"] == "delivered"
order_features["is_canceled"] = order_features["order_status"] == "canceled"
order_features["is_unavailable"] = order_features["order_status"] == "unavailable"

Orders where the two late-delivery definitions disagree: 0


## 9. Review features

For each order we keep the **latest** review: the one with the latest answer timestamp, then the latest creation date, then the highest `review_id` as a final tie-breaker so the result is reproducible. `review_count` keeps the number of review rows for the order.

Review features are known only **after** the review is written. They describe customer satisfaction after the fact and must not be used to explain an earlier contact.

In [11]:
review_count = reviews.groupby("order_id").size().rename("review_count").reset_index()

latest_reviews = reviews.sort_values(["order_id", "review_answer_timestamp", "review_creation_date", "review_id"],
                                     ascending=[True, False, False, False])
latest_reviews = latest_reviews.drop_duplicates(subset=["order_id"], keep="first")
latest_reviews = latest_reviews[["order_id", "review_id", "review_score", "review_creation_date", "review_answer_timestamp",
                                 "has_comment_title", "has_comment_message", "flag_repeated_review_id",
                                 "flag_review_created_before_purchase_date"]]
latest_reviews = latest_reviews.rename(columns={
    "has_comment_title": "review_has_title",
    "has_comment_message": "review_has_message",
    "flag_repeated_review_id": "review_flag_repeated_review_id",
    "flag_review_created_before_purchase_date": "review_flag_created_before_purchase_date",
})

order_features = order_features.merge(review_count, on="order_id", how="left", validate="one_to_one")
order_features = order_features.merge(latest_reviews, on="order_id", how="left", validate="one_to_one")
order_features["review_count"] = order_features["review_count"].fillna(0).astype(int)  # no review row = 0 reviews
order_features["review_is_low_score"] = (order_features["review_score"] <= 2).astype("boolean")
order_features.loc[order_features["review_score"].isna(), "review_is_low_score"] = pd.NA
order_features["delivery_to_review_days"] = (order_features["review_creation_date"]
                                             - order_features["order_delivered_customer_date"].dt.normalize()).dt.days
print("Rows after adding review features:", len(order_features))

display(order_features["review_count"].value_counts().sort_index().to_frame("orders"))
display(order_features["review_score"].value_counts(dropna=False).sort_index().to_frame("orders"))

Rows after adding review features: 99441


,orders
review_count,
0,768
1,98126
2,543
3,4


,orders
review_score,
1.0,11363
2.0,3131
3.0,8133
4.0,19038
5.0,57008
NaN,768


`review_count = 0` is a real count (no review row exists), so it is filled with 0. The other review columns stay empty for orders without a review. A negative `delivery_to_review_days` means the review was created before delivery; Olist sends its review request around the *estimated* delivery date, so this can happen for late orders.

## 10. Analysis window and censoring

### 10.1 Monthly order volume

The table shows why we use a window: 2016 has very few orders (and none in November), and September–October 2018 contain only a handful.

In [12]:
monthly_orders = order_features.groupby("purchase_year_month").size().rename("orders").to_frame()
display(monthly_orders.T)

purchase_year_month,2016-09,2016-10,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,...,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10
orders,4,324,1,800,1780,2682,2404,3700,3245,4026,...,7269,6728,7211,6939,6873,6167,6292,6512,16,4


### 10.2 Effective end of the data

The latest timestamp in the data (October 2018) is **not** the end of observation: after the end of August 2018 only a handful of stray events remain, while a normal weekday has around two hundred deliveries. Using the latest timestamp would wrongly treat orders still in transit at the end of August as "never delivered".

We therefore define the **effective data end** from delivery activity: the last day on which deliveries still reached at least 10% of the median weekday delivery count inside the analysis window. The 10% threshold is an assumption; the table below shows that the result does not depend much on its exact value, because activity drops from hundreds of deliveries a day to a few.

In [13]:
ANALYSIS_START = pd.Timestamp("2017-01-01")
ANALYSIS_END = pd.Timestamp("2018-08-31")
ACTIVITY_THRESHOLD_SHARE = 0.10  # assumption: "active" = at least 10% of a typical weekday

deliveries_per_day = order_features["order_delivered_customer_date"].dt.normalize().value_counts().sort_index()
in_window = (deliveries_per_day.index >= ANALYSIS_START) & (deliveries_per_day.index <= ANALYSIS_END)
is_weekday = deliveries_per_day.index.dayofweek < 5
typical_weekday_deliveries = deliveries_per_day[in_window & is_weekday].median()
activity_threshold = ACTIVITY_THRESHOLD_SHARE * typical_weekday_deliveries

active_days = deliveries_per_day[deliveries_per_day >= activity_threshold]
data_end_date = active_days.index.max()
milestone_columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
last_event_date = order_features[milestone_columns].max().max().normalize()

print("Median weekday deliveries in the window:", typical_weekday_deliveries)
print("Activity threshold (deliveries per day):", activity_threshold)
print("Effective data end:", data_end_date.date())
print("Latest timestamp of any order event:", last_event_date.date())
print("Latest review creation date:", order_features["review_creation_date"].max().date())

# Daily event counts around the effective data end
around_end = pd.DataFrame({"date": pd.date_range(data_end_date - pd.Timedelta(days=7), data_end_date + pd.Timedelta(days=7))})
for column in milestone_columns:
    counts_per_day = order_features[column].dropna().dt.normalize().value_counts()
    around_end[column] = around_end["date"].map(counts_per_day).fillna(0).astype(int)
display(around_end)

Median weekday deliveries in the window: 199.0
Activity threshold (deliveries per day): 19.900000000000002
Effective data end: 2018-08-31
Latest timestamp of any order event: 2018-10-17
Latest review creation date: 2018-08-31


,date,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,2018-08-24,99,138,199,356
1,2018-08-25,69,81,15,104
2,2018-08-26,73,70,0,7
3,2018-08-27,67,64,248,446
4,2018-08-28,44,56,118,395
5,2018-08-29,14,15,74,368
6,2018-08-30,4,0,2,365
7,2018-08-31,1,0,3,27
8,2018-09-01,0,0,0,6
9,2018-09-02,0,0,0,0


The table also shows that **purchases thin out during the last days of August 2018** (far below a normal day). Those days are inside the analysis window but incomplete; the forecasting phase should check where reliable purchase counts end before using the final week.

### 10.3 Window and end-of-data indicators

- `in_analysis_window`: purchase between 1 January 2017 and 31 August 2018 (inclusive).
- `delivery_outcome_observable`: `True` when we can tell whether the order arrived on time: it was delivered, or its estimated date had passed by the effective data end. `False` means the outcome is **censored**: the data stop before the order was due.
- `overdue_without_delivery`: the order is still in progress (status `shipped`, `invoiced`, `processing`, `created` or `approved`), has no delivery date, and its estimated date had passed by the effective data end. As far as the data show, these orders never arrived. They are a plausible trigger for future *simulated* "where is my order?" contacts. The 8 `delivered`-status orders without a delivery date are excluded, because their status contradicts the missing date (Phase 1C flag).

These three are hindsight indicators (`end_of_data`): useful for selecting records and as simulation inputs, not as features known at purchase.

In [14]:
order_features["in_analysis_window"] = order_features["purchase_date"].between(ANALYSIS_START, ANALYSIS_END)

has_delivery = order_features["has_customer_delivery_timestamp"]
estimate_has_passed = order_features["order_estimated_delivery_date"] < data_end_date
order_features["delivery_outcome_observable"] = has_delivery | estimate_has_passed

still_in_progress = order_features["order_status"].isin(["shipped", "invoiced", "processing", "created", "approved"])
order_features["overdue_without_delivery"] = still_in_progress & ~has_delivery & estimate_has_passed

window_summary = pd.DataFrame({
    "indicator": ["in_analysis_window", "delivery_outcome_observable", "overdue_without_delivery"],
    "true_all_orders": [int(order_features[c].sum()) for c in ["in_analysis_window", "delivery_outcome_observable", "overdue_without_delivery"]],
    "true_in_window": [int(order_features.loc[order_features["in_analysis_window"], c].sum())
                       for c in ["in_analysis_window", "delivery_outcome_observable", "overdue_without_delivery"]],
})
display(window_summary)
print("Orders outside the window:", int((~order_features["in_analysis_window"]).sum()))
print("Orders in the window with a censored delivery outcome:",
      int((order_features["in_analysis_window"] & ~order_features["delivery_outcome_observable"]).sum()))
display(order_features.loc[order_features["overdue_without_delivery"], "order_status"].value_counts().to_frame("overdue_orders"))

,indicator,true_all_orders,true_in_window
0,in_analysis_window,99092,99092
1,delivery_outcome_observable,99375,99046
2,overdue_without_delivery,1725,1696


Orders outside the window: 349
Orders in the window with a censored delivery outcome: 46


,overdue_orders
order_status,
shipped,1103
invoiced,314
processing,301
created,5
approved,2


## 11. Assemble the order table and the feature catalog

We now choose the final columns and their order, and write a **feature catalog**: one row per column with its meaning, its source and the stage at which it becomes known (`available_at`). The notebook stops if a column has no catalog entry, so nothing undocumented is exported.

The Phase 1C data-quality flags are carried over unchanged, so later phases can still filter on them.

In [15]:
# (column, available_at, source, description)
FEATURE_CATALOG = [
    ("order_id", "identifier", "orders", "Order identifier (one row per order)"),
    ("customer_id", "identifier", "orders", "Per-order customer identifier"),
    ("customer_unique_id", "identifier", "customers", "Person identifier; use for repeat customers"),
    ("order_status", "end_of_data", "orders", "Final order status at the end of the data"),
    ("order_purchase_timestamp", "purchase", "orders", "Purchase time"),
    ("order_approved_at", "approval", "orders", "Payment approval time"),
    ("order_delivered_carrier_date", "carrier_handoff", "orders", "Carrier handoff time"),
    ("order_delivered_customer_date", "delivery", "orders", "Customer delivery time"),
    ("order_estimated_delivery_date", "purchase", "orders", "Estimated delivery date promised at purchase"),
    ("customer_zip_code_prefix", "purchase", "customers", "Customer ZIP prefix (text)"),
    ("customer_city", "purchase", "customers", "Customer city"),
    ("customer_state", "purchase", "customers", "Customer state"),
    ("purchase_date", "purchase", "derived", "Calendar date of purchase"),
    ("purchase_year_month", "purchase", "derived", "Purchase month as YYYY-MM"),
    ("purchase_day_of_week", "purchase", "derived", "Day name of purchase"),
    ("purchase_hour", "purchase", "derived", "Hour of purchase (0-23)"),
    ("purchase_is_weekend", "purchase", "derived", "Purchase on Saturday or Sunday"),
    ("customer_order_number", "purchase", "derived", "Position of this order among the person's orders (1 = first)"),
    ("customer_is_repeat_buyer", "purchase", "derived", "Person had at least one earlier order"),
    ("item_count", "purchase", "order_items", "Number of item lines"),
    ("distinct_products", "purchase", "order_items", "Number of distinct products"),
    ("distinct_sellers", "purchase", "order_items", "Number of distinct sellers"),
    ("distinct_categories", "purchase", "order_items + products", "Number of distinct product categories"),
    ("items_price_total", "purchase", "order_items", "Sum of item prices (BRL)"),
    ("items_freight_total", "purchase", "order_items", "Sum of freight values (BRL)"),
    ("order_value", "purchase", "derived", "Price + freight (BRL)"),
    ("freight_share", "purchase", "derived", "Freight / order value"),
    ("items_weight_g_total", "purchase", "order_items + products", "Total product weight (g); empty if any item lacks weight"),
    ("latest_shipping_limit_date", "purchase", "order_items", "Latest seller shipping deadline among the items"),
    ("primary_category", "purchase", "products", "Category of the most expensive item (Portuguese)"),
    ("primary_category_english", "purchase", "category translation", "English category; empty if no translation"),
    ("primary_seller_id", "purchase", "order_items", "Seller of the most expensive item"),
    ("primary_seller_state", "purchase", "sellers", "State of the primary seller"),
    ("customer_seller_same_state", "purchase", "derived", "Customer and primary seller in the same state"),
    ("payment_count", "purchase", "payments", "Number of payment rows"),
    ("payment_total", "purchase", "payments", "Total amount paid (BRL)"),
    ("max_installments", "purchase", "payments", "Largest number of instalments"),
    ("uses_voucher", "purchase", "payments", "At least one voucher payment"),
    ("primary_payment_type", "purchase", "payments", "Type of the largest payment"),
    ("payment_minus_order_value", "purchase", "derived", "Payment total minus order value (BRL)"),
    ("purchase_to_approval_hours", "approval", "derived", "Hours from purchase to approval; empty if flagged"),
    ("approval_to_carrier_days", "carrier_handoff", "derived", "Days from approval to carrier; empty if flagged"),
    ("carrier_after_shipping_limit", "carrier_handoff", "derived", "Carrier handoff later than shipping deadline"),
    ("carrier_to_delivery_days", "delivery", "derived", "Days from carrier to customer; empty if flagged"),
    ("purchase_to_delivery_days", "delivery", "derived", "Days from purchase to delivery; empty if flagged"),
    ("promised_delivery_days", "purchase", "derived", "Calendar days between purchase and estimated date"),
    ("delivery_delay_days", "delivery", "derived", "Delivery date minus estimated date (negative = early)"),
    ("delivered_after_estimated_date", "delivery", "Phase 1C", "Delivered on a day after the estimated date"),
    ("is_delivered", "end_of_data", "derived", "Final status is delivered"),
    ("is_canceled", "end_of_data", "derived", "Final status is canceled"),
    ("is_unavailable", "end_of_data", "derived", "Final status is unavailable"),
    ("review_count", "review", "reviews", "Number of review rows for the order"),
    ("review_id", "review", "reviews", "ID of the latest review"),
    ("review_score", "review", "reviews", "Score of the latest review (1-5)"),
    ("review_is_low_score", "review", "derived", "Latest review score is 1 or 2"),
    ("review_creation_date", "review", "reviews", "Creation date of the latest review"),
    ("review_answer_timestamp", "review", "reviews", "Answer time of the latest review"),
    ("review_has_title", "review", "reviews", "Latest review has a title"),
    ("review_has_message", "review", "reviews", "Latest review has a message"),
    ("delivery_to_review_days", "review", "derived", "Review creation date minus delivery date"),
    ("review_flag_repeated_review_id", "review", "Phase 1C", "Latest review's review_id appears on several orders"),
    ("review_flag_created_before_purchase_date", "review", "Phase 1C", "Latest review created before purchase date"),
    ("in_analysis_window", "end_of_data", "derived", "Purchase between 2017-01-01 and 2018-08-31"),
    ("delivery_outcome_observable", "end_of_data", "derived", "On-time/late outcome known by the effective data end"),
    ("overdue_without_delivery", "end_of_data", "derived", "In-progress status, no delivery, estimate passed by effective data end"),
    ("has_approval_timestamp", "end_of_data", "Phase 1C", "Approval timestamp present"),
    ("has_carrier_handoff_timestamp", "end_of_data", "Phase 1C", "Carrier handoff timestamp present"),
    ("has_customer_delivery_timestamp", "end_of_data", "Phase 1C", "Delivery timestamp present"),
    ("has_items", "purchase", "Phase 1C", "Order has item rows"),
    ("has_payments", "purchase", "Phase 1C", "Order has payment rows"),
    ("has_reviews", "review", "Phase 1C", "Order has review rows"),
    ("flag_approved_before_purchase", "end_of_data", "Phase 1C", "Data-quality flag"),
    ("flag_carrier_before_approval", "end_of_data", "Phase 1C", "Data-quality flag"),
    ("flag_delivered_before_carrier", "end_of_data", "Phase 1C", "Data-quality flag"),
    ("flag_delivered_before_purchase", "end_of_data", "Phase 1C", "Data-quality flag"),
    ("flag_delivered_status_without_delivery_date", "end_of_data", "Phase 1C", "Data-quality flag"),
    ("flag_canceled_status_with_delivery_date", "end_of_data", "Phase 1C", "Data-quality flag"),
]

feature_catalog = pd.DataFrame(FEATURE_CATALOG, columns=["column", "available_at", "source", "description"])
final_columns = list(feature_catalog["column"])

undocumented = [column for column in order_features.columns if column not in final_columns]
if undocumented:
    raise ValueError(f"Columns without a catalog entry: {undocumented}. Add them to FEATURE_CATALOG.")
missing_from_table = [column for column in final_columns if column not in order_features.columns]
if missing_from_table:
    raise ValueError(f"Catalog lists columns that were never created: {missing_from_table}")

orders_analytical = order_features[final_columns].copy()
feature_catalog["dtype"] = [str(orders_analytical[column].dtype) for column in final_columns]
feature_catalog["missing_values"] = [int(orders_analytical[column].isna().sum()) for column in final_columns]

print("Order table:", orders_analytical.shape)
display(feature_catalog.groupby("available_at").size().rename("columns").to_frame())
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(feature_catalog)

Order table: (99441, 77)


,columns
available_at,
approval,2
carrier_handoff,3
delivery,5
end_of_data,16
identifier,3
purchase,36
review,12


,column,available_at,source,description,dtype,missing_values
0,order_id,identifier,orders,Order identifier (one row per order),object,0
1,customer_id,identifier,orders,Per-order customer identifier,object,0
2,customer_unique_id,identifier,customers,Person identifier; use for repeat customers,object,0
3,order_status,end_of_data,orders,Final order status at the end of the data,object,0
4,order_purchase_timestamp,purchase,orders,Purchase time,datetime64[ns],0
5,order_approved_at,approval,orders,Payment approval time,datetime64[ns],160
6,order_delivered_carrier_date,carrier_handoff,orders,Carrier handoff time,datetime64[ns],1783
7,order_delivered_customer_date,delivery,orders,Customer delivery time,datetime64[ns],2965
8,order_estimated_delivery_date,purchase,orders,Estimated delivery date promised at purchase,datetime64[ns],0
9,customer_zip_code_prefix,purchase,customers,Customer ZIP prefix (text),object,0


## 12. Validate the order table

The checks confirm that aggregation kept every order exactly once and lost no item, payment or review row.

In [16]:
checks = []
checks.append(("One row per order", len(orders_analytical) == len(orders)))
checks.append(("order_id is unique", orders_analytical["order_id"].is_unique))
checks.append(("Item rows = sum of item_count", int(orders_analytical["item_count"].sum()) == len(order_items)))
checks.append(("Payment rows = sum of payment_count", int(orders_analytical["payment_count"].sum()) == len(payments)))
checks.append(("Review rows = sum of review_count", int(orders_analytical["review_count"].sum()) == len(reviews)))
checks.append(("Item price total preserved", round(orders_analytical["items_price_total"].sum(), 2) == round(order_items["price"].sum(), 2)))
checks.append(("Payment total preserved", round(orders_analytical["payment_total"].sum(), 2) == round(payments["payment_value"].sum(), 2)))
checks.append(("item_count empty exactly when has_items is False",
               (orders_analytical["item_count"].isna() == ~orders_analytical["has_items"]).all()))
for column in ["purchase_to_approval_hours", "approval_to_carrier_days", "carrier_to_delivery_days", "purchase_to_delivery_days"]:
    checks.append((f"{column} has no negative values", bool((orders_analytical[column].dropna() >= 0).all())))

validation_df = pd.DataFrame(checks, columns=["check", "passed"])
display(validation_df)
if not validation_df["passed"].all():
    raise ValueError("An order-table validation check failed; see the table above.")

,check,passed
0,One row per order,True
1,order_id is unique,True
2,Item rows = sum of item_count,True
3,Payment rows = sum of payment_count,True
4,Review rows = sum of review_count,True
5,Item price total preserved,True
6,Payment total preserved,True
7,item_count empty exactly when has_items is False,True
8,purchase_to_approval_hours has no negative values,True
9,approval_to_carrier_days has no negative values,True


In [17]:
# Missing values in the exported order table, with the main reason
missing_table = feature_catalog.loc[feature_catalog["missing_values"] > 0, ["column", "available_at", "missing_values"]].copy()
missing_table["percent"] = (missing_table["missing_values"] / len(orders_analytical) * 100).round(2)
with pd.option_context("display.max_rows", None):
    display(missing_table.sort_values("missing_values", ascending=False))

,column,available_at,missing_values,percent
59,delivery_to_review_days,review,3611,3.63
41,approval_to_carrier_days,carrier_handoff,3156,3.17
43,carrier_to_delivery_days,delivery,2989,3.01
7,order_delivered_customer_date,delivery,2965,2.98
44,purchase_to_delivery_days,delivery,2965,2.98
46,delivery_delay_days,delivery,2965,2.98
47,delivered_after_estimated_date,delivery,2965,2.98
30,primary_category_english,purchase,2212,2.22
29,primary_category,purchase,2190,2.20
42,carrier_after_shipping_limit,carrier_handoff,1784,1.79


Missing values in the order table come from three sources, none of them imputed: an event that did not happen (no approval, carrier handoff, delivery or review), a related record that does not exist (775 orders without items, 1 without payments), and durations emptied because of a contradictory timestamp pair.

## 13. First look at the analytical dataset

A few summaries restricted to the analysis window. They are descriptive, not models, and show relationships that later phases may use when designing **simulated** contact scenarios.

In [18]:
window_orders = orders_analytical[orders_analytical["in_analysis_window"]]
delivered_orders = window_orders[window_orders["has_customer_delivery_timestamp"]]

monthly_delivery = delivered_orders.groupby("purchase_year_month").agg(
    delivered_orders=("order_id", "size"),
    late_orders=("delivered_after_estimated_date", "sum"),
    median_purchase_to_delivery_days=("purchase_to_delivery_days", "median"),
)
monthly_delivery["late_percent"] = (monthly_delivery["late_orders"] / monthly_delivery["delivered_orders"] * 100).round(2)
print("Late deliveries by purchase month (delivered orders in the window):")
display(monthly_delivery.round(2))

Late deliveries by purchase month (delivered orders in the window):


,delivered_orders,late_orders,median_purchase_to_delivery_days,late_percent
purchase_year_month,,,,
2017-01,750,22,10.73,2.93
2017-02,1653,49,10.97,2.96
2017-03,2546,116,10.13,4.56
2017-04,2303,151,12.75,6.56
2017-05,3545,106,9.79,2.99
2017-06,3135,95,10.80,3.03
2017-07,3872,108,10.16,2.79
2017-08,4193,122,9.77,2.91
2017-09,4150,182,10.31,4.39


In [19]:
reviewed_delivered = delivered_orders[delivered_orders["review_score"].notna()]
score_by_lateness = reviewed_delivered.groupby("delivered_after_estimated_date").agg(
    orders=("order_id", "size"),
    mean_review_score=("review_score", "mean"),
    low_score_percent=("review_is_low_score", "mean"),
)
score_by_lateness["low_score_percent"] = (score_by_lateness["low_score_percent"] * 100).round(2)
print("Latest review score by late delivery (delivered, reviewed orders in the window):")
display(score_by_lateness.round(2))

print("Final status of orders in the window:")
display(window_orders["order_status"].value_counts().to_frame("orders"))
print("Overdue orders without delivery in the window:", int(window_orders["overdue_without_delivery"].sum()))

Latest review score by late delivery (delivered, reviewed orders in the window):


,orders,mean_review_score,low_score_percent
delivered_after_estimated_date,,,
False,89182,4.29,9.25
True,6379,2.27,62.41


Final status of orders in the window:


,orders
order_status,
delivered,96211
shipped,1097
unavailable,602
canceled,580
processing,299
invoiced,296
created,5
approved,2


Overdue orders without delivery in the window: 1696


Late delivery is associated with much lower review scores. This is a **correlation in post-purchase feedback**, not evidence of customer contacts; it motivates, but does not measure, a late-delivery contact scenario.

## 14. Daily activity table

One row per calendar day, from the first purchase date to the last recorded order event, including days with no events. Days after the effective data end are kept so the totals reconcile with the order table, and marked with `after_effective_data_end`. Each column counts e-commerce events by the **date on which the event happened**:

| Column | Counted on the date of |
|---|---|
| `orders_purchased` | purchase |
| `orders_approved` | approval |
| `orders_handed_to_carrier` | carrier handoff |
| `orders_delivered` | delivery |
| `late_deliveries` | delivery, for orders delivered after the estimated date |
| `deliveries_due` | estimated delivery date |
| `canceled_orders_by_purchase_date` | purchase, for canceled orders (Olist has no cancellation date) |
| `reviews_created` | review creation, counting each distinct `review_id` once |
| `low_score_reviews` | review creation, distinct reviews with score 1 or 2 |
| `order_value_purchased` | purchase; sum of order values (BRL) |

**None of these columns is a contact count.** They are candidate drivers for contact demand, to be linked to contacts only through explicit assumptions in a later phase.

In [20]:
first_day = orders_analytical["purchase_date"].min()
daily_activity = pd.DataFrame({"date": pd.date_range(first_day, last_event_date, freq="D")})

distinct_reviews = reviews.drop_duplicates(subset=["review_id"])
late_mask = orders_analytical["delivered_after_estimated_date"].fillna(False).astype(bool)

daily_event_dates = {
    "orders_purchased": orders_analytical["order_purchase_timestamp"],
    "orders_approved": orders_analytical["order_approved_at"],
    "orders_handed_to_carrier": orders_analytical["order_delivered_carrier_date"],
    "orders_delivered": orders_analytical["order_delivered_customer_date"],
    "late_deliveries": orders_analytical.loc[late_mask, "order_delivered_customer_date"],
    "deliveries_due": orders_analytical["order_estimated_delivery_date"],
    "canceled_orders_by_purchase_date": orders_analytical.loc[orders_analytical["is_canceled"], "order_purchase_timestamp"],
    "reviews_created": distinct_reviews["review_creation_date"],
    "low_score_reviews": distinct_reviews.loc[distinct_reviews["review_score"] <= 2, "review_creation_date"],
}

for column, event_dates in daily_event_dates.items():
    counts_per_day = event_dates.dropna().dt.normalize().value_counts()
    daily_activity[column] = daily_activity["date"].map(counts_per_day).fillna(0).astype(int)

value_per_day = orders_analytical.groupby("purchase_date")["order_value"].sum()
daily_activity["order_value_purchased"] = daily_activity["date"].map(value_per_day).fillna(0).round(2)

daily_activity["day_of_week"] = daily_activity["date"].dt.day_name()
daily_activity["is_weekend"] = daily_activity["date"].dt.dayofweek >= 5
daily_activity["in_analysis_window"] = daily_activity["date"].between(ANALYSIS_START, ANALYSIS_END)
daily_activity["after_effective_data_end"] = daily_activity["date"] > data_end_date

print("Daily table:", daily_activity.shape, "from", daily_activity["date"].min().date(), "to", daily_activity["date"].max().date())
display(daily_activity[daily_activity["in_analysis_window"]].describe().round(1))

Daily table: (774, 15) from 2016-09-04 to 2018-10-17


,date,orders_purchased,orders_approved,orders_handed_to_carrier,orders_delivered,late_deliveries,deliveries_due,canceled_orders_by_purchase_date,reviews_created,low_score_reviews,order_value_purchased
count,608,608.0,608.0,608.0,608.0,608.0,608.0,608.0,608.0,608.0,608.0
mean,2017-10-31 12:00:00,163.0,162.8,160.2,158.1,10.6,159.6,1.0,161.3,23.5,25964.2
min,2017-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,2017-06-01 18:00:00,103.0,94.0,8.0,37.0,1.0,0.0,0.0,43.0,7.0,15690.9
50%,2017-10-31 12:00:00,150.5,145.0,153.0,143.0,5.0,147.0,1.0,148.5,20.0,24029.2
75%,2018-04-01 06:00:00,218.0,218.0,267.0,268.2,12.2,277.0,1.0,265.0,32.0,34434.8
max,2018-08-31 00:00:00,1176.0,990.0,707.0,446.0,90.0,522.0,9.0,461.0,112.0,178377.6
std,NaN,91.2,103.8,142.3,124.9,15.4,156.9,1.2,124.6,21.6,14621.0


Events dated after the last day of the table are not counted: a few estimated delivery dates lie beyond the last recorded order event, so `deliveries_due` sums to slightly less than the number of orders. The validation below accounts for this.

In [21]:
daily_checks = []
expected_days = (last_event_date - first_day).days + 1
daily_checks.append(("One row per day, no gaps", len(daily_activity) == expected_days and daily_activity["date"].is_unique))
for column, event_dates in daily_event_dates.items():
    # Expected total: events dated on or before the last day of the table
    expected_total = int((event_dates.dropna().dt.normalize() <= last_event_date).sum())
    daily_checks.append((f"{column} total matches source", int(daily_activity[column].sum()) == expected_total))
daily_checks.append(("orders_purchased equals number of orders", int(daily_activity["orders_purchased"].sum()) == len(orders_analytical)))

daily_validation = pd.DataFrame(daily_checks, columns=["check", "passed"])
display(daily_validation)
if not daily_validation["passed"].all():
    raise ValueError("A daily-table validation check failed; see the table above.")

print("Estimated delivery dates after the last day of the table (not counted):",
      int((orders_analytical["order_estimated_delivery_date"] > last_event_date).sum()))
print("Days after the effective data end:", int(daily_activity["after_effective_data_end"].sum()))

,check,passed
0,"One row per day, no gaps",True
1,orders_purchased total matches source,True
2,orders_approved total matches source,True
3,orders_handed_to_carrier total matches source,True
4,orders_delivered total matches source,True
5,late_deliveries total matches source,True
6,deliveries_due total matches source,True
7,canceled_orders_by_purchase_date total matches...,True
8,reviews_created total matches source,True
9,low_score_reviews total matches source,True


Estimated delivery dates after the last day of the table (not counted): 6
Days after the effective data end: 47


In [22]:
# Weekly pattern of purchases inside the window
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
window_days = daily_activity[daily_activity["in_analysis_window"]]
weekday_pattern = window_days.groupby("day_of_week")[["orders_purchased", "orders_delivered", "reviews_created"]].mean()
display(weekday_pattern.reindex(weekday_order).round(1))

print("Busiest purchase days in the window:")
display(window_days.nlargest(5, "orders_purchased")[["date", "day_of_week", "orders_purchased", "order_value_purchased"]])

,orders_purchased,orders_delivered,reviews_created
day_of_week,,,
Monday,185.5,216.1,12.4
Tuesday,182.7,212.1,206.2
Wednesday,178.2,207.0,217.5
Thursday,169.0,195.7,213.7
Friday,161.8,198.0,202.1
Saturday,126.1,63.2,204.4
Sunday,137.1,13.8,73.5


Busiest purchase days in the window:


,date,day_of_week,orders_purchased,order_value_purchased
446,2017-11-24,Friday,1176,178377.63
447,2017-11-25,Saturday,499,71858.16
449,2017-11-27,Monday,403,56644.85
448,2017-11-26,Sunday,391,53595.20
450,2017-11-28,Tuesday,380,56248.47


## 15. Export

Same conventions as Phase 1C: Parquet in `data/processed/`, which keeps datetime, nullable boolean and text types. The feature catalog is also saved as CSV so it can be read without Python.

In [23]:
orders_path = PROCESSED_DIR / "olist_orders_analytical.parquet"
daily_path = PROCESSED_DIR / "olist_daily_activity.parquet"
catalog_path = PROCESSED_DIR / "olist_feature_catalog.csv"

orders_analytical.to_parquet(orders_path, index=False)
daily_activity.to_parquet(daily_path, index=False)
feature_catalog.to_csv(catalog_path, index=False)

export_check = []
for path, table in [(orders_path, orders_analytical), (daily_path, daily_activity)]:
    reloaded = pd.read_parquet(path)
    export_check.append({"file": path.name, "rows": len(reloaded), "columns": len(reloaded.columns),
                         "same_rows": len(reloaded) == len(table), "same_types": reloaded.dtypes.equals(table.dtypes),
                         "size_mb": round(path.stat().st_size / 1024**2, 2)})
export_check.append({"file": catalog_path.name, "rows": len(feature_catalog), "columns": len(feature_catalog.columns),
                     "same_rows": True, "same_types": True, "size_mb": round(catalog_path.stat().st_size / 1024**2, 3)})
display(pd.DataFrame(export_check))

,file,rows,columns,same_rows,same_types,size_mb
0,olist_orders_analytical.parquet,99441,77,True,True,24.490
1,olist_daily_activity.parquet,774,15,True,True,0.030
2,olist_feature_catalog.csv,77,6,True,True,0.006


## 16. Summary and limitations

**Datasets produced** (in `data/processed/`)

- `olist_orders_analytical.parquet`: one row per order, with order, customer, item, payment, delivery-timeline and latest-review features, analysis-window and censoring indicators, and the Phase 1C quality flags.
- `olist_daily_activity.parquet`: one row per day with e-commerce event counts, calendar fields and the window indicator.
- `olist_feature_catalog.csv`: meaning, source, `available_at` stage, type and missing count of every order-table column.

**Decisions applied**

- No order removed; the analysis window is a flag (`in_analysis_window`), not a filter.
- Durations are empty when their timestamp pair was flagged as contradictory; nothing was imputed.
- The latest review represents an order; `review_count` keeps the number of review rows.
- Primary category and seller come from the most expensive item line.
- Untranslated categories keep an empty English name.

**How to use the features**

- Respect `available_at`: a simulated contact may only use features known before the contact time. Review features and `end_of_data` indicators are hindsight.
- Use `delivery_outcome_observable` to avoid treating censored orders at the end of the data as on time. The effective data end is derived from delivery activity (section 10.2), not from the latest stray timestamp.
- Purchase counts thin out in the last days of August 2018; check them before using the end of the window for forecasting.
- Use `customer_unique_id` for anything about people.

**Limitations**

- Olist contains **no contacts, contact reasons, channels, waiting times, handling times, or agents**. The daily table counts orders, deliveries and reviews, which are *candidate drivers* of contact demand, not contact demand.
- The relationship between late delivery and low review scores is descriptive.
- The data cover about 20 usable months, which limits seasonal modelling (only one November/Black Friday inside the window).
- Cancellation dates are not recorded; cancellations can only be dated by purchase date.

> **Synthetic contact records will be generated in a later phase** from these datasets and documented assumptions. They must always be kept separate from, and never presented as, observed Olist data.